In [14]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

# 1️⃣ Chrome binary yolu (normal Chrome)
chrome_path = r"C:\Program Files\Google\Chrome\Application\chrome.exe"  # kendi bilgisayara göre değiştir

# 2️⃣ ChromeDriver yolu
driver_path = r"C:\Program Files\Google\Chrome\Application\chromedriver-win64\chromedriver.exe"  # chromedriver.exe’nin tam yolu

# 3️⃣ Chrome Options
options = Options()
options.binary_location = chrome_path

# 4️⃣ Service
service = Service(driver_path)

# 5️⃣ WebDriver başlat
driver = webdriver.Chrome(service=service, options=options)

# 6️⃣ Test: Google aç
driver.get("https://www.google.com")

print("Chrome Selenium ile açıldı ✅")


Chrome Selenium ile açıldı ✅


In [37]:
import json
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import time

chrome_path = r"C:\Program Files\Google\Chrome\Application\chrome.exe"
driver_path = r"C:\Program Files\Google\Chrome\Application\chromedriver-win64\chromedriver.exe"

options = Options()
options.binary_location = chrome_path
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

service = Service(driver_path)
driver = webdriver.Chrome(service=service, options=options)

url = "https://www.arabam.com/ilan/galeriden-satilik-opel-corsa-1-4-120-yil/120-yila-ozel-paket-dusuk-km-otomatik-corsa/28013982"
driver.get(url)
time.sleep(5)

arac_bilgileri = {}

# Basliktan arac bilgileri (önceki regex ile)
try:
    baslik = driver.find_element(By.XPATH, "//h1").text.strip()
except:
    baslik = ""

pattern = r"(Galeriden|Sahibinden)\s+(\w+)\s+(\w+)\s+([\w\. ]+)\s+(\d{4})\s+Model\s+\w+\s+([\d\.]+ km)\s+(\w+)"
match = re.search(pattern, baslik)
if match:
    arac_bilgileri["Kimden"] = match.group(1)
    arac_bilgileri["Marka"] = match.group(2)
    arac_bilgileri["Seri"] = match.group(3)
    arac_bilgileri["Model"] = match.group(4).strip()
    arac_bilgileri["Yıl"] = match.group(5)
    arac_bilgileri["Kilometre"] = match.group(6)
    arac_bilgileri["Renk"] = match.group(7)

# Fiyat
try:
    fiyat_raw = driver.find_element(By.XPATH, "//div[contains(@class,'price')]").text.strip()
    fiyat_sayi = re.sub(r"[^\d]", "", fiyat_raw)
    if fiyat_sayi:
        arac_bilgileri["Fiyat"] = f"{fiyat_sayi} TL"
except:
    arac_bilgileri["Fiyat"] = None

# Sayfa kaynağındaki JSON'u bul
page_source = driver.page_source

# Örnek: JSON içinde "ValueDescription" geçen tüm parçaları bul
parca_pattern = r'\"([^\"]+)\",\"Value\"\:\"[^\"]*\".*?\"ValueDescription\"\:\"([^\"]+)\"'
parcalar = re.findall(parca_pattern, page_source)

for parca_adi, durum in parcalar:
    arac_bilgileri[parca_adi] = durum

# CSV Kaydet
df = pd.DataFrame([arac_bilgileri])
df.to_csv("arabam_ilani_parca_json.csv", index=False, encoding="utf-8-sig")

print("✅ Araç + Parça bilgileri JSON’dan alındı ve CSV’ye kaydedildi!")

driver.quit()


✅ Araç + Parça bilgileri JSON’dan alındı ve CSV’ye kaydedildi!


In [4]:
import json
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time
import random

# ======= Ayarlar =======
chrome_path = r"C:\Program Files\Google\Chrome\Application\chrome.exe"
driver_path = r"C:\Program Files\Google\Chrome\Application\chromedriver-win64\chromedriver.exe"
base_url = "https://www.arabam.com/ikinci-el/otomobil/dacia?page={}"  # Sayfa linki
toplam_sayfa = 40  # Kaç sayfa taranacak

# ======= Chrome Başlat =======
options = Options()
options.binary_location = chrome_path
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
service = Service(driver_path)
driver = webdriver.Chrome(service=service, options=options)

# ======= Sonuçları saklayacak liste =======
tum_araclar = []

for sayfa in range(1, toplam_sayfa + 1):
    print(f"➡️ Sayfa {sayfa} kontrol ediliyor...")
    driver.get(base_url.format(sayfa))
    time.sleep(random.uniform(3, 6))  # Sayfa yüklenme beklemesi
    
    # İlan linklerini al
    try:
        ilanlar = driver.find_elements(By.XPATH, "//a[contains(@href,'/ilan/')]")
        ilan_linkleri = [i.get_attribute("href") for i in ilanlar]
        ilan_linkleri = list(dict.fromkeys(ilan_linkleri))  # tekrar edenleri çıkar
    except:
        ilan_linkleri = []

    if not ilan_linkleri:
        print("   0 ilan bulundu.")
        continue

    for link in ilan_linkleri:
        arac_bilgileri = {}
        driver.get(link)
        time.sleep(random.uniform(2, 5))  # ilan sayfası beklemesi

        # Başlık ve temel bilgiler
        try:
            baslik = driver.find_element(By.XPATH, "//h1").text.strip()
            pattern = r"(Galeriden|Sahibinden)\s+(\w+)\s+(\w+)\s+([\w\. ]+)\s+(\d{4})\s+Model\s+\w+\s+([\d\.]+ km)\s+(\w+)"
            match = re.search(pattern, baslik)
            if match:
                arac_bilgileri["Kimden"] = match.group(1)
                arac_bilgileri["Marka"] = match.group(2)
                arac_bilgileri["Seri"] = match.group(3)
                arac_bilgileri["Model"] = match.group(4).strip()
                arac_bilgileri["Yıl"] = match.group(5)
                arac_bilgileri["Kilometre"] = match.group(6)
                arac_bilgileri["Renk"] = match.group(7)
        except:
            pass

        # Fiyat
        try:
            fiyat_raw = driver.find_element(By.XPATH, "//div[contains(@class,'price')]").text.strip()
            fiyat_sayi = re.sub(r"[^\d]", "", fiyat_raw)
            if fiyat_sayi:
                arac_bilgileri["Fiyat"] = f"{fiyat_sayi} TL"
        except:
            arac_bilgileri["Fiyat"] = None

        # Parça bilgileri (JSON’dan)
        try:
            time.sleep(random.uniform(1, 3))  # JS yüklenmesi için bekleme
            page_source = driver.page_source
            # JSON içindeki tüm parçaları ve durumlarını yakala
            parca_pattern = r'"([^"]+?)","Value":.*?"ValueDescription":"(.*?)"'
            parcalar = re.findall(parca_pattern, page_source)
            for parca, durum in parcalar:
                arac_bilgileri[parca] = durum
        except:
            pass

        tum_araclar.append(arac_bilgileri)

# ======= CSV Kaydet =======
df = pd.DataFrame(tum_araclar)
df.to_csv("7.dacia.csv", index=False, encoding="utf-8-sig")
print("✅ Tüm ilanlar ve parça bilgileri başarıyla kaydedildi!")

driver.quit()


➡️ Sayfa 1 kontrol ediliyor...
➡️ Sayfa 2 kontrol ediliyor...
➡️ Sayfa 3 kontrol ediliyor...
➡️ Sayfa 4 kontrol ediliyor...
➡️ Sayfa 5 kontrol ediliyor...
➡️ Sayfa 6 kontrol ediliyor...
➡️ Sayfa 7 kontrol ediliyor...
➡️ Sayfa 8 kontrol ediliyor...
➡️ Sayfa 9 kontrol ediliyor...
➡️ Sayfa 10 kontrol ediliyor...
➡️ Sayfa 11 kontrol ediliyor...
➡️ Sayfa 12 kontrol ediliyor...
➡️ Sayfa 13 kontrol ediliyor...
➡️ Sayfa 14 kontrol ediliyor...
➡️ Sayfa 15 kontrol ediliyor...
➡️ Sayfa 16 kontrol ediliyor...
➡️ Sayfa 17 kontrol ediliyor...
➡️ Sayfa 18 kontrol ediliyor...
➡️ Sayfa 19 kontrol ediliyor...
➡️ Sayfa 20 kontrol ediliyor...
➡️ Sayfa 21 kontrol ediliyor...
➡️ Sayfa 22 kontrol ediliyor...
➡️ Sayfa 23 kontrol ediliyor...
➡️ Sayfa 24 kontrol ediliyor...
➡️ Sayfa 25 kontrol ediliyor...
➡️ Sayfa 26 kontrol ediliyor...
➡️ Sayfa 27 kontrol ediliyor...
➡️ Sayfa 28 kontrol ediliyor...
➡️ Sayfa 29 kontrol ediliyor...
➡️ Sayfa 30 kontrol ediliyor...
➡️ Sayfa 31 kontrol ediliyor...
➡️ Sayfa 32 kontr